# Pitch cards: each team A track as a frozen frame with the pitch around the player

For the Pass Review app's "Team A · on the pitch" cards. One small picture per team A track (the moment the
player is biggest on screen), with a white box on the player. Output per segment in
`MyDrive/Playbook/review/match_video_2/<segment>/`: `pitch.jpg` and `pitch.json`.

**Runtime:** CPU is enough, about 1 minute per segment. **Runtime -> Run all**; do not edit cells.
If it disconnects, **Run all again**: finished segments are skipped.

In [ ]:
# Cell 1 — Drive, inputs, our repo
import os, json, glob, shutil, subprocess, sys, time
from google.colab import drive
drive.mount('/content/drive')
PBD = '/content/drive/MyDrive/Playbook'
MATCH = 'match_video_2'
CLIPS = sorted(glob.glob(f'{PBD}/video_downloads/{MATCH}_clips/*.mp4'))
REVIEW = f'{PBD}/review/{MATCH}'
PB, BRANCH = '/content/playbook', 'claude/setup-gpu-video-testing-JhgUH'
if not os.path.isdir(PB + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/MuwafagQ/Playbook-program.git', PB], check=True)
else:
    subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', BRANCH], cwd=PB, check=True)
    subprocess.run(['git', 'reset', '-q', '--hard', 'FETCH_HEAD'], cwd=PB, check=True)
CARDS = json.load(open(f'{PB}/data/review/pitch_cards_mv2.json'))
if not CLIPS:
    raise SystemExit('The match clips are not in Drive.')
print(len(CLIPS), 'segments,', sum(map(len, CARDS.values())), 'cards')

In [ ]:
# Cell 2 — Cut the cards from the video (skips finished segments)
import numpy as np, cv2
CW, CH, COLS = 320, 180, 6
for clip in CLIPS:
    n = os.path.basename(clip)[:-4]
    if n not in CARDS or os.path.exists(f'{REVIEW}/{n}/pitch.json'):
        continue
    t0 = time.time(); todo = {}
    for q, f, x1, y1, x2, y2 in CARDS[n]:
        todo.setdefault(f, []).append((q, x1, y1, x2, y2))
    cards = {}
    cap = cv2.VideoCapture(clip); fi = 0; last = max(todo)
    while fi <= last:
        if not cap.grab():
            break
        if fi in todo:
            ok, fr = cap.retrieve(); H, W = fr.shape[:2]
            for q, x1, y1, x2, y2 in todo[fi]:
                bh = y2 - y1; cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                sw = min(W, max(480, bh * 6)); sh = sw * 9 / 16
                sx = int(max(0, min(W - sw, cx - sw / 2))); sy = int(max(0, min(H - sh, cy - sh / 2)))
                crop = fr[sy:sy + int(sh), sx:sx + int(sw)].copy()
                k = CW / crop.shape[1]
                im = cv2.resize(crop, (CW, CH), interpolation=cv2.INTER_AREA)
                cv2.rectangle(im, (int((x1 - sx) * k) - 2, int((y1 - sy) * k) - 2), (int((x2 - sx) * k) + 2, int((y2 - sy) * k) + 2), (255, 255, 255), 2)
                cards[q] = im
        fi += 1
    cap.release()
    ids = [c[0] for c in CARDS[n] if c[0] in cards]
    rows = (len(ids) + COLS - 1) // COLS
    sheet = np.zeros((max(1, rows) * CH, COLS * CW, 3), np.uint8); index = {}
    for i, q in enumerate(ids):
        x, y = (i % COLS) * CW, (i // COLS) * CH
        sheet[y:y + CH, x:x + CW] = cards[q]; index[str(q)] = [x, y]
    cv2.imwrite('/content/pitch.jpg', sheet, [cv2.IMWRITE_JPEG_QUALITY, 78])
    shutil.copy('/content/pitch.jpg', f'{REVIEW}/{n}/pitch.jpg')
    json.dump({'w': CW, 'h': CH, 'pieces': index}, open(f'{REVIEW}/{n}/pitch.json', 'w'))
    print(f'{n}: {len(ids)} cards, {(time.time() - t0) / 60:.1f} min')
print(f'Done. Tell Claude the pitch cards are in {REVIEW}.')